In [16]:
%pip install stanza

  Obtaining dependency information for stanza from https://files.pythonhosted.org/packages/ac/f6/c50d0ee85e40687a81a95d90d426938d0d83ac0683efa2b87fa4110b665c/stanza-1.10.1-py3-none-any.whl.metadata
  Obtaining dependency information for emoji from https://files.pythonhosted.org/packages/91/db/a0335710caaa6d0aebdaa65ad4df789c15d89b7babd9a30277838a7d9aac/emoji-2.14.1-py3-none-any.whl.metadata
  Obtaining dependency information for protobuf>=3.15.0 from https://files.pythonhosted.org/packages/cc/5b/0d421533c59c789e9c9894683efac582c06246bf24bb26b753b149bd88e4/protobuf-6.32.0-cp39-abi3-macosx_10_9_universal2.whl.metadata
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/eb/8d/776adee7bbf76365fdd7f2552710282c79a4ead5d2a46408c9043a2b70ba/networkx-3.5-py3-none-any.whl.metadata
  Obtaining dependency information for torch>=1.3.0 from https://files.pythonhosted.org/packages/be/66/5c9a321b325aaecb92d4d1855421e3a055abd77903b7dab6575ca07796db/torch-2.8.0-c

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import stanza
from collections import Counter
from itertools import islice


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Load datasets
fake_df = pd.read_csv("../data/Fake.csv")
true_df = pd.read_csv("../data/True.csv")

# Check the first few rows
print("Fake news dataset:")
print(fake_df.head())

print("\nTrue news dataset:")
print(true_df.head())

print(true_df.shape, fake_df.shape)


Fake news dataset:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  

True news dataset:
                                               title  \
0  As U.S. budget fight looms, Republicans fl

In [10]:
fake_df.info()
true_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [11]:
true_df.nunique()
fake_df.nunique()

title      17903
text       17455
subject        6
date        1681
dtype: int64

In [12]:
true_df.describe(include='all')
fake_df.describe(include='all')

,title,text,subject,date
count,23481,23481,23481,23481
unique,17903,17455,6,1681
top,MEDIA IGNORES Time That Bill Clinton FIRED His...,,News,"May 10, 2017"
freq,6,626,9050,46


In [13]:
true_df.isnull().sum()
fake_df.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

In [19]:
import stanza
from collections import Counter
from itertools import islice
import numpy as np

# --- configure Stanza ---
LANG = "en"              # change if needed
stanza.download(LANG)    # run once per language
nlp = stanza.Pipeline(LANG, processors="tokenize,pos,lemma", tokenize_no_ssplit=True, use_gpu=True)

def _ngrams(seq, n):
    return zip(*(islice(seq, i, None) for i in range(n)))

def tokenize_subjects(subjects, keep_upos={"NOUN","PROPN","ADJ","VERB"}, use_lemmas=True):
    """
    subjects: iterable[str]
    returns: list[list[str]] tokens per subject
    """
    tokenized = []
    for s in subjects:
        doc = nlp(s if isinstance(s, str) else str(s))
        toks = []
        for sent in doc.sentences:
            for w in sent.words:
                if w.upos == "PUNCT":
                    continue
                if keep_upos and w.upos not in keep_upos:
                    continue
                tok = (w.lemma if use_lemmas else w.text).lower()
                toks.append(tok)
        tokenized.append(toks)
    return tokenized

def analyze_subjects(df, label, top_k=20):
    from collections import Counter
    subjects = df['subject'].astype(str).fillna("").tolist()
    tokens_per_subject = tokenize_subjects(subjects)

    # word counts
    word_counts = Counter()
    for toks in tokens_per_subject:
        word_counts.update(toks)

    # bigrams & trigrams
    bigrams = Counter()
    trigrams = Counter()
    for toks in tokens_per_subject:
        bigrams.update(_ngrams(toks, 2))
        trigrams.update(_ngrams(toks, 3))
    bigrams = Counter({" ".join(k): v for k, v in bigrams.items()})
    trigrams = Counter({" ".join(k): v for k, v in trigrams.items()})

    lengths = [len(toks) for toks in tokens_per_subject]
    stats = {
        "dataset": label,
        "rows": len(df),
        "unique_subject_strings": df['subject'].nunique(),
        "mean_tokens_per_subject": float(np.mean(lengths)) if lengths else 0.0,
        "median_tokens_per_subject": float(np.median(lengths)) if lengths else 0.0,
        "type_token_ratio": (len(word_counts) / max(sum(lengths), 1)) if lengths else 0.0,
        "top_subject_strings": df['subject'].value_counts().head(top_k),
        "top_words": word_counts.most_common(top_k),
        "top_bigrams": bigrams.most_common(top_k),
        "top_trigrams": trigrams.most_common(top_k),
        "word_counts": word_counts,
        "bigram_counts": bigrams,
        "trigram_counts": trigrams,
    }
    return stats

# Run your analysis
true_stats = analyze_subjects(true_df, "TRUE")
fake_stats = analyze_subjects(fake_df, "FAKE")

def print_stats(stats):
    print(f"\n=== {stats['dataset']} dataset ===")
    print(f"Rows: {stats['rows']}")
    print(f"Unique subject strings: {stats['unique_subject_strings']}")
    print(f"Mean tokens/subject: {stats['mean_tokens_per_subject']:.2f}")
    print(f"Median tokens/subject: {stats['median_tokens_per_subject']:.0f}")
    print(f"Type/Token ratio: {stats['type_token_ratio']:.3f}")

    print("\nTop subject strings:")
    for s, c in stats["top_subject_strings"].items():
        print(f"  {c:>6}  {s}")

    print("\nTop words (lemmas):")
    for w, c in stats["top_words"]:
        print(f"  {c:>6}  {w}")

    print("\nTop bigrams:")
    for g, c in stats["top_bigrams"]:
        print(f"  {c:>6}  {g}")

    print("\nTop trigrams:")
    for g, c in stats["top_trigrams"]:
        print(f"  {c:>6}  {g}")

print_stats(true_stats)
print_stats(fake_stats)

# Optional: distinctive words comparison (same as before)
def log_odds_ratio(counts_A, counts_B, alpha=0.01, top_k=20):
    import numpy as np
    vocab = set(counts_A) | set(counts_B)
    A = np.array([counts_A.get(t, 0) + alpha for t in vocab], dtype=float)
    B = np.array([counts_B.get(t, 0) + alpha for t in vocab], dtype=float)
    A_tot, B_tot = A.sum(), B.sum()
    logit_A = np.log(A / (A_tot - A))
    logit_B = np.log(B / (B_tot - B))
    delta = logit_A - logit_B
    var = 1/A + 1/B
    z = delta / np.sqrt(var)
    terms = list(vocab)
    top_A = sorted(zip(terms, z), key=lambda x: -x[1])[:top_k]
    top_B = sorted(zip(terms, z), key=lambda x: x[1])[:top_k]
    return top_A, top_B

print("\n=== Distinctive words (TRUE vs FAKE) ===")
top_true, top_fake = log_odds_ratio(true_stats["word_counts"], fake_stats["word_counts"])
print("\nMore distinctive for TRUE:")
for t, z in top_true: print(f"  {z:>7.2f}  {t}")
print("\nMore distinctive for FAKE:")
for t, z in top_fake: print(f"  {z:>7.2f}  {t}")

2025-09-09 15:42:15 INFO: Downloaded file to /Users/alexandrud.soroiu/stanza_resources/resources.json
2025-09-09 15:42:15 INFO: Downloading default packages for language: en (English) ...
2025-09-09 15:42:16 INFO: File exists: /Users/alexandrud.soroiu/stanza_resources/en/default.zip
2025-09-09 15:42:18 INFO: Finished downloading models and saved to /Users/alexandrud.soroiu/stanza_resources
2025-09-09 15:42:18 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-09-09 15:42:18 INFO: Downloaded file to /Users/alexandrud.soroiu/stanza_resources/resources.json
2025-09-09 15:42:18 WARNING: Language en package default expects mwt, which has been added
2025-09-09 15:42:18 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined 


=== TRUE dataset ===
Rows: 21417
Unique subject strings: 2
Mean tokens/subject: 1.00
Median tokens/subject: 1
Type/Token ratio: 0.000

Top subject strings:
   11272  politicsNews
   10145  worldnews

Top words (lemmas):
   11272  politicsnew
   10145  worldnew

Top bigrams:

Top trigrams:

=== FAKE dataset ===
Rows: 23481
Unique subject strings: 6
Mean tokens/subject: 1.29
Median tokens/subject: 1
Type/Token ratio: 0.000

Top subject strings:
    9050  News
    6841  politics
    4459  left-news
    1570  Government News
     783  US_News
     778  Middle-east

Top words (lemmas):
   15079  news
    6841  politics
    4459  left
    1570  government
     783  us_news
     778  middle
     778  east

Top bigrams:
    4459  left news
    1570  government news
     778  middle east

Top trigrams:

=== Distinctive words (TRUE vs FAKE) ===

More distinctive for TRUE:
     1.50  politicsnew
     1.48  worldnew
    -1.09  east
    -1.09  middle
    -1.09  us_news
    -1.17  government
    -1